### Import packages

In [1]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path

### Select a subset of chunked_paragraphs_with_embeddings df to test uncertainty words from financial dictionary

In [9]:
chunked_pg_w_embed = pd.read_csv("data_files/chunked_paragraphs_with_embeddings.csv")

chunked_pg_w_embed.head(5)

,report_name,paragraph_number,text,subcategory,row_id,match_method,matched_phrase,category,component
0,2000-01-01__12069__The Budget and Economic Out...,1,FISCAL YEARS 2001-2010 The Congress of the Uni...,NaN,0,NaN,NaN,NaN,NaN
1,2000-01-01__12069__The Budget and Economic Out...,2,Some figures in this report indicate periods o...,NaN,1,NaN,NaN,NaN,NaN
2,2000-01-01__12069__The Budget and Economic Out...,3,Numbers in the text and tables may not add up ...,NaN,2,NaN,NaN,NaN,NaN
3,2000-01-01__12069__The Budget and Economic Out...,4,"ERRATA In the PDF, PostScript, and WordPerfect...",NaN,3,NaN,NaN,NaN,NaN
4,2000-01-01__12069__The Budget and Economic Out...,5,Preface T his volume is one of a series of rep...,NaN,4,NaN,NaN,NaN,NaN


In [14]:
chunked_pg_w_embed_20 = chunked_pg_w_embed.dropna(subset=['match_method', 'matched_phrase', 'category', 'subcategory', 'component']).head(20)

### Import the financial dictionary

In [25]:
# Load the Loughran-McDonald dictionary
lm_df = pd.read_csv('data_files/Loughran-McDonald_MasterDictionary_1993-2025.csv')

# Ensure the 'Word' column is string and uppercase (LM dictionary defaults to uppercase)
lm_df['Word'] = lm_df['Word'].astype(str).str.upper()

### Measure uncertainty in the 20 paragraphs

In [26]:
# Create sets of uncertain to track
uncertain_words = set(lm_df[lm_df['Uncertainty'] > 0]['Word'])

# Define the scoring function
def compute_uncertainty_score(paragraph_text):
    """
    Takes a paragraph text, counts occurrences of uncertain words, 
    and divides by the total word count of the paragraph.
    Returns a float value between 0 and 1.
    """
    # Handle empty or non-string inputs safely
    if not isinstance(paragraph_text, str) or not paragraph_text.strip():
        return 0.0
    
    # Clean and tokenize text into individual words (ignoring punctuation/numbers)
    # Convert to uppercase to match the dictionary format
    words = re.findall(r'\b[A-Za-z]+\b', paragraph_text.upper())
    total_words = len(words)
    
    # Avoid division by zero if the paragraph contains no valid words
    if total_words == 0:
        return 0.0
    
    # Count how many words in the paragraph exist in our uncertainty set
    uncertain_count = sum(1 for word in words if word in uncertain_words)
    
    # Compute the ratio (value between 0 and 1)
    uncertainty_score = uncertain_count / total_words
    
    return uncertainty_score

In [37]:
chunked_pg_w_embed_20['uncertainty_score'] = chunked_pg_w_embed_20['text'].apply(compute_uncertainty_score)

# Preview the results
print(f"The total number of uncertainty words from the Loughran-McDonald dictionary is: {len(uncertain_words)}")
chunked_pg_w_embed_20[['text', 'uncertainty_score']].head(5)

The total number of uncertainty words from the Loughran-McDonald dictionary is: 297


,text,uncertainty_score
7,An early version of the economic forecast unde...,0.000000
9,Dan L. Crippen Director January 2000 This stud...,0.009654
11,"In the case of discretionary spending, however...",0.000000
12,Without any definitive answer to those questio...,0.076923
16,The Congress has used each of those spending p...,0.015038


### Measure uncertainty of paragraphs in the chunked_pg_w_embed df with non-null match_method, matched_phrase, category, subcategory, component columns

In [40]:
chunked_pg_w_embed = chunked_pg_w_embed.dropna(subset=['match_method', 'matched_phrase', 'category', 'subcategory', 'component'])

In [41]:
chunked_pg_w_embed['uncertainty_score'] = chunked_pg_w_embed['text'].apply(compute_uncertainty_score)

C:\Users\trung\AppData\Local\Temp\ipykernel_41824\3248015158.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunked_pg_w_embed['uncertainty_score'] = chunked_pg_w_embed['text'].apply(compute_uncertainty_score)


In [43]:
# get descriptive statistics for uncertainty score
chunked_pg_w_embed['uncertainty_score'].describe()

count    14989.000000
mean         0.008251
std          0.013539
min          0.000000
25%          0.000000
50%          0.000000
75%          0.012346
max          0.151515
Name: uncertainty_score, dtype: float64

In [50]:
# export chunked_pg_w_embed with uncertainty score
# chunked_pg_w_embed.to_csv("data_files/chunked_paragraphs_with_embeddings_uncert.csv", index=False)